In [ ]:
import duckdb
import matplotlib.pyplot as plt

In [ ]:
con = duckdb.connect()
pv = con.execute("""
    SELECT time, generation_mw, capacity_mwp
    FROM read_parquet('data/bronze/pv_live/entity_type=pes/region_id=20/*.parquet')
    WHERE time >= '2025-06-02' AND time < '2025-06-05'
    ORDER BY time
""").df()

wx = con.execute("""
    SELECT time, shortwave_radiation
    FROM read_parquet('data/bronze/weather/source=era5/lead_days=0/region_id=20/*.parquet')
    WHERE time >= '2025-06-02' AND time < '2025-06-05'
    ORDER BY time
""").df()

In [ ]:
print(pv.describe())
print("peak gen / capacity:", pv.generation_mw.max() / pv.capacity_mwp.max())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(pv.time, pv.generation_mw / pv.capacity_mwp, label="generation / capacity")
ax.plot(wx.time, wx.shortwave_radiation / wx.shortwave_radiation.max(), label="GHI (normalised)")
ax.legend()
ax.grid(alpha=0.3)
plt.savefig("scratch_sanity.png", dpi=120, bbox_inches="tight")

In [ ]:
import pandas as pd

g = (pv.set_index("time").generation_mw / pv.set_index("time").capacity_mwp).rename("gen")
w = wx.set_index("time").shortwave_radiation.resample("30min").interpolate().rename("ghi")
j = pd.concat([g, w], axis=1).dropna()

for k in range(-4, 5):
    print(f"{k*30:+5d} min   r = {j.gen.corr(j.ghi.shift(k)):.4f}")